# Course 5: NLP Applications
## Lecture 2: Text Summarization & AI Text Generation Overview

---

### 🎯 Learning Objectives:
By the end of this lecture notebook, you will be able to:
1. Distinguish between **Extractive** and **Abstractive** text summarization.
2. Implement transformer-based abstractive summarization using Hugging Face pipelines (`t5-small`, `facebook/bart-large-cnn`).
3. Understand **Autoregressive Text Generation** and master decoding parameters: **Temperature, Top-K, Top-P (Nucleus Sampling), and Beam Search**.
4. Evaluate generated text quantitatively using **ROUGE (ROUGE-1, ROUGE-2, ROUGE-L)** and **BLEU** metrics.
5. Handle practical production challenges like document chunking for long texts.

---

### 📖 Summarization Paradigms

```
┌────────────────────────────────────────────────────────────────────────┐
│                          Long Source Document                          │
└────────────────────────────────────────────────────────────────────────┘
                                    │
           ┌────────────────────────┴────────────────────────┐
           ▼                                                 ▼
┌─────────────────────────────────────┐   ┌─────────────────────────────────────┐
│      Extractive Summarization       │   │      Abstractive Summarization      │
│  - Selects key sentences directly   │   │  - Generates novel paraphrased text │
│  - Fast, preserves verbatim facts   │   │  - Uses Sequence-to-Sequence models │
│  - Algorithms: TextRank, TF-IDF     │   │  - Models: T5, BART, Pegasus, LLMs  │
└─────────────────────────────────────┘   └─────────────────────────────────────┘
```


In [ ]:
# ==========================================
# Step 0: Imports & Setup
# ==========================================
import warnings
warnings.filterwarnings('ignore')
import re
import numpy as np
import pandas as pd
from transformers import pipeline, set_seed

# Optional rouge_score for evaluation
try:
    from rouge_score import rouge_scorer
    print("✅ rouge-score is ready!")
except ImportError:
    print("⚠️ rouge-score not found. Installing...")
    # !pip install rouge-score


---
## Section 1: Extractive Summarization Baseline (TextRank / Frequency)

Extractive summarization scores sentences based on keyword frequencies or graph centrality and extracts top sentences without modifying words.


In [ ]:
# ==========================================
# Step 1: Lightweight Extractive Summarizer
# ==========================================
import collections

sample_long_article = """
Artificial intelligence and natural language processing have transformed how businesses process unstructured text.
In customer support, automated systems can triage thousands of tickets every hour by identifying urgency and topic.
Furthermore, text summarization allows managers to review lengthy email threads and meeting transcripts in seconds.
Modern summarization models rely on transformer architectures such as BART and T5 which understand complex semantics.
However, extractive methods remain useful in low-resource environments where computational power is strictly limited.
Despite these advances, generative models can occasionally hallucinate facts that were not present in the original text.
Therefore, deploying NLP applications requires careful validation, safety guardrails, and human-in-the-loop oversight.
"""

def simple_extractive_summary(text: str, num_sentences: int = 2) -> str:
    """Scores sentences based on word frequencies and picks the top N sentences."""
    sentences = [s.strip() for s in text.strip().split('.') if s.strip()]
    words = re.findall(r'\w+', text.lower())
    
    # Filter common stop words
    stopwords = set(["and", "the", "in", "to", "of", "a", "is", "that", "it", "on", "for", "as", "with", "these"])
    filtered_words = [w for w in words if w not in stopwords]
    word_freq = collections.Counter(filtered_words)
    
    # Score each sentence
    sentence_scores = {}
    for i, sent in enumerate(sentences):
        sent_words = re.findall(r'\w+', sent.lower())
        score = sum(word_freq[w] for w in sent_words if w in word_freq)
        sentence_scores[i] = score / max(1, len(sent_words))
    
    # Select top ranked sentences preserving original order
    top_indices = sorted(sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:num_sentences])
    summary = ". ".join([sentences[i] for i in top_indices]) + "."
    return summary

ext_summary = simple_extractive_summary(sample_long_article, num_sentences=2)
print("📄 Extractive Summary Output:")
print(ext_summary)


---
## Section 2: Abstractive Summarization with Hugging Face Transformers

Abstractive summarization uses an **Encoder-Decoder** model (like T5 or BART) to comprehend the context and write a coherent, condensed summary in fresh words.


In [ ]:
# ==========================================
# Step 2: Hugging Face Abstractive Summarization
# ==========================================
# We use t5-small for fast execution or facebook/bart-large-cnn for high quality
summarizer = pipeline(
    "summarization", 
    model="t5-small"
)

article_to_summarize = """
The Apollo program was the third United States human spaceflight program carried out by the National Aeronautics 
and Space Administration (NASA), which succeeded in landing the first humans on the Moon in 1969. First conceived 
during the presidency of Dwight D. Eisenhower, Apollo was dedicated to John F. Kennedy's national goal of landing 
a man on the Moon and returning him safely to the Earth before the decade of the 1960s was out. 
The goal was accomplished on July 20, 1969, when American astronauts Neil Armstrong and Buzz Aldrin landed the 
Apollo Lunar Module Eagle on the Moon, while Michael Collins remained in lunar orbit in the command and service module. 
All three returned safely to Earth on July 24.
"""

summary_result = summarizer(
    article_to_summarize, 
    max_length=60, 
    min_length=20, 
    do_sample=False,
    num_beams=4
)

print("📝 Source Text Word Count:", len(article_to_summarize.split()))
print("✨ Abstractive Summary:", summary_result[0]['summary_text'])
print("📊 Summary Word Count:", len(summary_result[0]['summary_text'].split()))


---
## Section 3: AI Text Generation & Decoding Strategies

Autoregressive models (like GPT-2, LLaMA, Mistral) generate text token-by-token predicting $P(w_t \mid w_1, \dots, w_{t-1})$.

### Key Generation Hyperparameters:
1. **Greedy Search**: Always selects the highest probability token. Can lead to repetitive, monotonous text.
2. **Beam Search**: Keeps top $B$ candidate hypotheses at each step. Great for translation and summarization.
3. **Temperature ($T$)**: Modulates the probability distribution over vocabulary:
   $$P(w_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$
   - $T < 1.0$: Confident, conservative, focused text.
   - $T > 1.0$: Creative, diverse, but potentially erratic text.
4. **Top-K Sampling**: Restricts sampling to the top $K$ most likely tokens.
5. **Top-P (Nucleus Sampling)**: Dynamically selects the smallest set of tokens whose cumulative probability exceeds $P$ (e.g. 0.90).


In [ ]:
# ==========================================
# Step 3: Comparing Text Generation Strategies
# ==========================================
generator = pipeline("text-generation", model="distilgpt2")
set_seed(42)

prompt = "In the future, artificial intelligence in healthcare will"

print(f"🎯 Base Prompt: '{prompt}'\n")

# Strategy 1: Greedy Decoding
greedy_out = generator(prompt, max_length=50, do_sample=False)[0]['generated_text']
print("1️⃣ Greedy Output:")
print(f"   {greedy_out}\n")

# Strategy 2: Creative Sampling (High Temperature + Top-P)
creative_out = generator(
    prompt, 
    max_length=50, 
    do_sample=True, 
    temperature=0.9, 
    top_k=50, 
    top_p=0.92
)[0]['generated_text']
print("2️⃣ Creative Sampling (Temp=0.9, Top-P=0.92):")
print(f"   {creative_out}\n")

# Strategy 3: Controlled Sampling (Low Temperature)
focused_out = generator(
    prompt, 
    max_length=50, 
    do_sample=True, 
    temperature=0.3, 
    top_p=0.85
)[0]['generated_text']
print("3️⃣ Focused Sampling (Temp=0.3, Top-P=0.85):")
print(f"   {focused_out}")


---
## Section 4: Quantitative Evaluation with ROUGE & BLEU

How do we measure if a generated summary is good?
- **ROUGE-1**: Overlap of unigrams (single words) between generated summary and human reference.
- **ROUGE-2**: Overlap of bigrams (pairs of words).
- **ROUGE-L**: Measures the **Longest Common Subsequence (LCS)**, rewarding sentence structure similarity.


In [ ]:
# ==========================================
# Step 4: ROUGE Evaluation Metric in Python
# ==========================================
reference_summary = "NASA landed the first humans on the Moon in July 1969 during the Apollo 11 mission."
generated_summary = summary_result[0]['summary_text']

try:
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference_summary, generated_summary)
    
    print("📊 ROUGE Evaluation Results:")
    print("--------------------------------------------------")
    print(f"Reference: {reference_summary}")
    print(f"Generated: {generated_summary}\n")
    for metric, score in scores.items():
        print(f"{metric.upper():<8} -> Precision: {score.precision:.3f} | Recall: {score.recall:.3f} | F1: {score.fmeasure:.3f}")
    print("--------------------------------------------------")
except Exception as e:
    print("Rouge calculation error (rouge-score may not be installed):", e)


---
## Section 5: Chunking Strategy for Long Documents

Transformer models have a fixed maximum sequence length (e.g. 512 tokens for T5, 1024 for BART). To summarize long documents:
1. Split document into semantic chunks (e.g., 400 words per chunk).
2. Summarize each chunk individually.
3. Combine individual summaries and run a final consolidation pass ("Map-Reduce Summarization").


In [ ]:
# ==========================================
# Step 5: Map-Reduce Chunking Summarizer Function
# ==========================================
def summarize_long_document(text: str, max_chunk_words: int = 250) -> str:
    """Splits long text into manageable chunks and summarizes each."""
    words = text.split()
    if len(words) <= max_chunk_words:
        res = summarizer(text, max_length=70, min_length=20, do_sample=False)
        return res[0]['summary_text']
    
    # Split into word chunks
    chunks = [" ".join(words[i:i + max_chunk_words]) for i in range(0, len(words), max_chunk_words)]
    print(f"📦 Document partitioned into {len(chunks)} chunks for summarization...")
    
    chunk_summaries = []
    for idx, chunk in enumerate(chunks):
        res = summarizer(chunk, max_length=50, min_length=15, do_sample=False)
        chunk_summaries.append(res[0]['summary_text'])
    
    combined_intermediate = " ".join(chunk_summaries)
    
    # Final condensation pass
    final_summary = summarizer(combined_intermediate, max_length=80, min_length=30, do_sample=False)
    return final_summary[0]['summary_text']

# Quick test with duplicated text
long_test_text = (article_to_summarize + " ") * 3
final_res = summarize_long_document(long_test_text)
print("🎯 Final Consolidated Summary:")
print(final_res)
